In [1]:
import pandas as pd
import numpy as np 
import seaborn as sns 
import matplotlib.pyplot as plt 

In [2]:
df = pd.read_csv("..\data\RawData\customer_churn_dataset.csv")

In [3]:
df.head()

,CustomerID,Age,Gender,Tenure,Usage Frequency,Support Calls,Payment Delay,Subscription Type,Contract Length,Total Spend,Last Interaction,Churn
0,2.0,30.0,Female,39.0,14.0,5.0,18.0,Standard,Annual,932.0,17.0,1.0
1,3.0,65.0,Female,49.0,1.0,10.0,8.0,Basic,Monthly,557.0,6.0,1.0
2,4.0,55.0,Female,14.0,4.0,6.0,18.0,Basic,Quarterly,185.0,3.0,1.0
3,5.0,58.0,Male,38.0,21.0,7.0,7.0,Standard,Monthly,NaN,29.0,1.0
4,6.0,23.0,Male,32.0,20.0,5.0,8.0,Basic,Monthly,617.0,20.0,1.0


In [4]:
df.shape

(440833, 12)

In [5]:
df.columns

Index(['CustomerID', 'Age', 'Gender', 'Tenure', 'Usage Frequency',
       'Support Calls', 'Payment Delay', 'Subscription Type',
       'Contract Length', 'Total Spend', 'Last Interaction', 'Churn'],
      dtype='str')

In [6]:
df.info()
df.isnull().sum()

<class 'pandas.DataFrame'>
RangeIndex: 440833 entries, 0 to 440832
Data columns (total 12 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   CustomerID         440832 non-null  float64
 1   Age                440827 non-null  float64
 2   Gender             440832 non-null  str    
 3   Tenure             440829 non-null  float64
 4   Usage Frequency    440826 non-null  float64
 5   Support Calls      440829 non-null  float64
 6   Payment Delay      440830 non-null  float64
 7   Subscription Type  440830 non-null  str    
 8   Contract Length    440831 non-null  str    
 9   Total Spend        440827 non-null  float64
 10  Last Interaction   440831 non-null  float64
 11  Churn              440832 non-null  float64
dtypes: float64(9), str(3)
memory usage: 48.5 MB


CustomerID           1
Age                  6
Gender               1
Tenure               4
Usage Frequency      7
Support Calls        4
Payment Delay        3
Subscription Type    3
Contract Length      2
Total Spend          6
Last Interaction     2
Churn                1
dtype: int64

In [7]:
num_cols = df.select_dtypes(include=['int64','float64']).columns
cat_cols = df.select_dtypes(include=['object']).columns

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_14248\3927880714.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include=['object']).columns


In [8]:
for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

In [9]:
for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

In [10]:
df.isnull().sum()

CustomerID           0
Age                  0
Gender               0
Tenure               0
Usage Frequency      0
Support Calls        0
Payment Delay        0
Subscription Type    0
Contract Length      0
Total Spend          0
Last Interaction     0
Churn                0
dtype: int64

* Numerical features were imputed using median to reduce impact of outliers, while categorical features were imputed using mode to preserve distribution.

* **avg_spend**
* This feature captures how much a customer spends relative to their duration with the company. It helps identify high-value customers and normalizes spending behavior across customers with different tenures.

In [11]:
df['avg_spend'] = df['Total Spend'] / (df['Tenure']+1)
# Capture customer value

* **engagement sore**
* This feature measures overall customer engagement by combining usage frequency and duration. Higher engagement typically indicates stronger customer loyalty and lower likelihood of churn.

In [12]:
df['engagement_score'] = df['Usage Frequency'] * df['Tenure']
# High engagement = low churn usually

* **support_pressure**
* This feature represents how frequently a customer contacts support. A higher value may indicate dissatisfaction or recurring issues, which are strong indicators of potential churn.

In [13]:
df['support_pressure'] = df['Support Calls']/(df['Tenure']+1)
# More complaints = dangerous customer

* **payment_risk**
* This feature combines financial behavior (payment delays) with customer dissatisfaction (support calls). Customers with high values are more likely to churn due to both payment issues and service dissatisfaction.

In [14]:
df['payment_risk'] = df['Payment Delay']*df['Support Calls']
# Delays + complaints = dangerous customer

* **high value**
* This feature combines financial behavior (payment delays) with customer dissatisfaction (support calls). Customers with high values are more likely to churn due to both payment issues and service dissatisfaction.

In [15]:
df['high_value'] = (df['Total Spend'] > df['Total Spend'].median()).astype(int)

* **inactive risk**
* This feature acts as a proxy for customer inactivity. Higher values indicate that the customer has not interacted recently, which increases the likelihood of churn.

In [16]:
df['inactive_risk'] = df['Last Interaction']
# Higher = more likely churn

* **tenure group**
* This feature categorizes customers based on their tenure. Different tenure groups exhibit different churn patterns—for example, new customers are generally more likely to churn compared to long-term customers.

In [17]:
df['tenure_group'] = pd.cut(
    df['Tenure'],
    bins=[0,12,24,48,100],
    labels = ['0-1yr','1-2yr','2-4yr','4+yr']
)

#### Encoding Categorical columns

In [18]:
df['Gender'] = df['Gender'].map({'Male':1,'Female':0})

In [19]:
df = pd.get_dummies(df, columns=['Subscription Type'], drop_first=True)

In [20]:
df = pd.get_dummies(df, columns=['Contract Length'], drop_first=True)

#### Drop Useless columns

In [21]:
df = df.drop(columns=['CustomerID'])

In [22]:
df.shape

(440833, 20)

In [23]:
df.head()

,Age,Gender,Tenure,Usage Frequency,Support Calls,Payment Delay,Total Spend,Last Interaction,Churn,avg_spend,engagement_score,support_pressure,payment_risk,high_value,inactive_risk,tenure_group,Subscription Type_Premium,Subscription Type_Standard,Contract Length_Monthly,Contract Length_Quarterly
0,30.0,0,39.0,14.0,5.0,18.0,932.0,17.0,1.0,23.300000,546.0,0.125000,90.0,1,17.0,2-4yr,False,True,False,False
1,65.0,0,49.0,1.0,10.0,8.0,557.0,6.0,1.0,11.140000,49.0,0.200000,80.0,0,6.0,4+yr,False,False,True,False
2,55.0,0,14.0,4.0,6.0,18.0,185.0,3.0,1.0,12.333333,56.0,0.400000,108.0,0,3.0,1-2yr,False,False,False,True
3,58.0,1,38.0,21.0,7.0,7.0,661.0,29.0,1.0,16.948718,798.0,0.179487,49.0,0,29.0,2-4yr,False,True,True,False
4,23.0,1,32.0,20.0,5.0,8.0,617.0,20.0,1.0,18.696970,640.0,0.151515,40.0,0,20.0,2-4yr,False,False,True,False


In [24]:
bool_cols = df.select_dtypes(include=['bool']).columns

df[bool_cols] = df[bool_cols].astype(int)

### **EDA**

In [25]:
df['Churn'].value_counts()
# to check how data are balance ot imbalance.

Churn
1.0    250000
0.0    190833
Name: count, dtype: int64

In [26]:
df.groupby('Churn')[['Tenure','Total Spend','Usage Frequency',
    'Support Calls','Payment Delay','inactive_risk']].mean()

,Tenure,Total Spend,Usage Frequency,Support Calls,Payment Delay,inactive_risk
Churn,,,,,,
0.0,32.281702,749.953246,16.260484,1.586418,10.015500,13.008866
1.0,30.473580,541.289383,15.461660,5.144796,15.217684,15.604540


* Customers who churn tend to have lower tenure, lower engagement, higher payment delays, indicating dissatisfaction and lower loyalty.

In [27]:
pd.crosstab(df['Gender'],df['Churn'])
# is churn different by gender

Churn,0.0,1.0
Gender,,
0,63522,127058
1,127311,122942


* There are some columns which have >70% correlation. 
    * inactive risk and Last interaction have 100% positive correlation.
    * Total spend and high_value have 82% positive correlation.
    * Payment_risk and support calls have 75% positive correlation.
* so Last Interaction and high_value can be drop because of it give >80% same information like inactive risk and total spend.

In [32]:
df = df.drop(columns=['Last Interaction','high_value','Support Calls','Payment Delay'], errors='ignore')

In [33]:
df.columns

Index(['Age', 'Gender', 'Tenure', 'Usage Frequency', 'Total Spend', 'Churn',
       'avg_spend', 'engagement_score', 'support_pressure', 'payment_risk',
       'inactive_risk', 'tenure_group', 'Subscription Type_Premium',
       'Subscription Type_Standard', 'Contract Length_Monthly',
       'Contract Length_Quarterly'],
      dtype='str')

In [34]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 440833 entries, 0 to 440832
Data columns (total 16 columns):
 #   Column                      Non-Null Count   Dtype   
---  ------                      --------------   -----   
 0   Age                         440833 non-null  float64 
 1   Gender                      440833 non-null  int64   
 2   Tenure                      440833 non-null  float64 
 3   Usage Frequency             440833 non-null  float64 
 4   Total Spend                 440833 non-null  float64 
 5   Churn                       440833 non-null  float64 
 6   avg_spend                   440833 non-null  float64 
 7   engagement_score            440833 non-null  float64 
 8   support_pressure            440833 non-null  float64 
 9   payment_risk                440833 non-null  float64 
 10  inactive_risk               440833 non-null  float64 
 11  tenure_group                440833 non-null  category
 12  Subscription Type_Premium   440833 non-null  int64   
 13  Subscripti

In [35]:
df = pd.get_dummies(df,drop_first=True)

In [36]:
df.describe()

,Age,Gender,Tenure,Usage Frequency,Total Spend,Churn,avg_spend,engagement_score,support_pressure,payment_risk,inactive_risk,Subscription Type_Premium,Subscription Type_Standard,Contract Length_Monthly,Contract Length_Quarterly
count,440833.000000,440833.000000,440833.000000,440833.000000,440833.000000,440833.000000,440833.000000,440833.000000,440833.000000,440833.000000,440833.000000,440833.000000,440833.000000,440833.000000,440833.000000
mean,39.373223,0.567682,31.256301,15.807465,631.618263,0.567108,36.802875,490.111834,0.217486,50.863193,14.480894,0.337266,0.338291,0.197587,0.400446
std,12.442323,0.495399,17.255685,8.586179,240.801804,0.495477,53.698193,405.522394,0.427803,62.991601,8.596177,0.472777,0.473129,0.398180,0.489989
min,18.000000,0.000000,1.000000,1.000000,100.000000,0.000000,1.639344,1.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000
25%,29.000000,0.000000,16.000000,9.000000,480.000000,0.000000,12.770755,156.000000,0.033333,6.000000,7.000000,0.000000,0.000000,0.000000,0.000000
50%,39.000000,1.000000,32.000000,16.000000,661.000000,1.000000,19.440000,380.000000,0.094340,25.000000,14.000000,0.000000,0.000000,0.000000,0.000000
75%,48.000000,1.000000,46.000000,23.000000,830.000000,1.000000,35.985714,738.000000,0.217391,72.000000,22.000000,1.000000,1.000000,0.000000,1.000000
max,65.000000,1.000000,60.000000,30.000000,1000.000000,1.000000,500.000000,1800.000000,5.000000,300.000000,30.000000,1.000000,1.000000,1.000000,1.000000


In [37]:
df.to_csv("../data/Processed/churn_cleaned.csv", index=False)